# CE49X: Introduction to Computational Thinking and Data Science for Civil Engineers
## Week 5: In-Class Exercise — Are Cold Winters Disappearing?

**Instructor:** Dr. Eyuphan Koc  
**Department of Civil Engineering, Bogazici University**  
**Semester:** Spring 2026

---

> *"In lecture, we investigated whether hot extremes in August are getting more extreme. Now let's flip the question: are cold winters disappearing?"*

You will apply the **same statistical methodology** — descriptive statistics, distributions, z-scores, and hypothesis testing — to **January temperatures** from the same dataset. Same tools, fresh angle, independent findings.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
%matplotlib inline

# Display settings
pd.set_option('display.max_columns', 10)
pd.set_option('display.width', 120)
plt.rcParams['figure.dpi'] = 100

In [ ]:
# Load and prepare data (same as lecture)
df_raw = pd.read_csv('data/GlobalLandTemperaturesByMajorCity.csv')

# Parse dates and extract time columns
df_raw['dt'] = pd.to_datetime(df_raw['dt'])
df_raw['year'] = df_raw['dt'].dt.year
df_raw['month'] = df_raw['dt'].dt.month

# Select the same 8 target cities
target_cities = ['Istanbul', 'London', 'New York', 'Tokyo',
                 'Sydney', 'Cairo', 'São Paulo', 'Bombay']

display_names = {'Bombay': 'Mumbai', 'São Paulo': 'São Paulo',
                 'New York': 'New York', 'Istanbul': 'Istanbul',
                 'London': 'London', 'Tokyo': 'Tokyo',
                 'Sydney': 'Sydney', 'Cairo': 'Cairo'}

df = df_raw[df_raw['City'].isin(target_cities)].copy()
df['CityLabel'] = df['City'].map(display_names)

# Filter to 1850-2013 (reliable coverage) and drop missing values
df = df[(df['year'] >= 1850) & (df['year'] <= 2013)].copy()
df = df.dropna(subset=['AverageTemperature'])

print(f'Dataset ready: {len(df):,} rows across {df["City"].nunique()} cities')
print(f'Year range: {df["year"].min()}–{df["year"].max()}')

In [ ]:
# Quick check — confirm the data looks right
print(f'Shape: {df.shape}')
df.head()

---
## Task 1: Descriptive Statistics [PRACTICE]

> **Your Task:**  
> 1. Filter the data to **January only** (`month == 1`).  
> 2. Compute descriptive statistics per city: **mean, std, min, max** of `AverageTemperature`.  
> 3. Display the result as a table, sorted by mean temperature.
>
> **Hint:** Use `df[df['month'] == 1].groupby('CityLabel')['AverageTemperature'].agg(...)` with a list of functions.

In [ ]:
df_jan = df[df['month'] == 1]
stats_jan = df_jan.groupby('CityLabel')['AverageTemperature'].agg(['mean', 'std', 'min', 'max'])
stats_jan = stats_jan.sort_values('mean')
stats_jan

> **Interpretation:**  
> Which city has the **lowest** average January temperature? Which has the **highest variability** (standard deviation)? Why might that be — think about geography and climate type.

**Answer:** 
- **New York** has the lowest average January temperature (approx -3.04°C).
- **New York** also has the highest variability (std dev approx 2.54°C), followed closely by Istanbul (2.02°C).
- This high variability in New York and Istanbul is typical of continental and temperate climates where seasonal changes are more pronounced and influenced by shifting air masses, compared to tropical cities like Mumbai which are very stable (std 0.69°C).

---
## Task 2: Visualize Early vs. Recent [PRACTICE]

> **Your Task (Part A — Histograms):**  
> 1. Filter to **January** data for **Istanbul**.  
> 2. Split into two periods: **early (1900–1950)** and **recent (1980–2013)**.  
> 3. Create **side-by-side histograms** (use `fig, (ax1, ax2) = plt.subplots(1, 2, ...)`) comparing the two periods.  
> 4. Use the same x-axis range for both so they are directly comparable.
>
> **Hint:** Set `bins=12` and use colors `steelblue` (early) and `indianred` (recent). Add `ax.grid(True, alpha=0.3)` to both.

In [ ]:
ist_jan = df_jan[df_jan['CityLabel'] == 'Istanbul']
early = ist_jan[(ist_jan['year'] >= 1900) & (ist_jan['year'] <= 1950)]['AverageTemperature']
recent = ist_jan[(ist_jan['year'] >= 1980) & (ist_jan['year'] <= 2013)]['AverageTemperature']

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5), sharex=True)
ax1.hist(early, bins=12, color='steelblue', edgecolor='black', alpha=0.7)
ax1.set_title('Istanbul January: 1900-1950')
ax1.grid(True, alpha=0.3)

ax2.hist(recent, bins=12, color='indianred', edgecolor='black', alpha=0.7)
ax2.set_title('Istanbul January: 1980-2013')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

> **Your Task (Part B — Boxplots by Decade):**  
> 1. Still using Istanbul January data, create a **decade** column: `decade = (year // 10) * 10`.  
> 2. Filter to decades from 1900 onward.  
> 3. Create a **boxplot** showing January temperature distributions by decade.
>
> **Hint:** Use `ax.boxplot(...)` or `df.boxplot(column=..., by=...)`. Label axes clearly.

In [ ]:
ist_jan_dec = ist_jan[ist_jan['year'] >= 1900].copy()
ist_jan_dec['decade'] = (ist_jan_dec['year'] // 10) * 10

fig, ax = plt.subplots(figsize=(10, 6))
ist_jan_dec.boxplot(column='AverageTemperature', by='decade', ax=ax)
ax.set_title('Istanbul January Temperatures by Decade')
ax.set_xlabel('Decade')
ax.set_ylabel('Temperature (°C)')
plt.suptitle('') # Remove automatic title
plt.show()

> **Interpretation:**  
> Describe what you see. Has the distribution of January temperatures in Istanbul **shifted**? In which direction? Do you see any change in the **spread** (variability) over time?

**Answer:** 
- There is a slight **upward shift** in the median January temperature in Istanbul, especially visible in the more recent decades (1980s-2010s) compared to the early 20th century.
- The spread (variability) remains significant across all decades, which is typical for Istanbul's climate, but the lower bound (minimums) appears to be trending slightly higher, suggesting that extremely cold Januaries are becoming less frequent or less extreme.

---
## Task 3: Z-Score Analysis [PRACTICE]

> **Your Task (Part A — Compute Z-Scores):**  
> 1. Filter to **January** data only.  
> 2. Using **1900–1950 as the baseline period**, compute the mean and standard deviation of January temperatures for **each city** separately.  
> 3. Calculate z-scores for **all** January temperatures using each city's baseline: $z = (x - \mu_{\text{baseline}}) / \sigma_{\text{baseline}}$  
>
> **Hint:** Use `groupby('CityLabel')` on the baseline subset to get per-city mean/std, then merge or map back.

In [ ]:
baseline = df_jan[(df_jan['year'] >= 1900) & (df_jan['year'] <= 1950)]
baseline_stats = baseline.groupby('CityLabel')['AverageTemperature'].agg(['mean', 'std']).reset_index()
baseline_stats.columns = ['CityLabel', 'baseline_mean', 'baseline_std']

df_jan_z = df_jan.merge(baseline_stats, on='CityLabel')
df_jan_z['zscore'] = (df_jan_z['AverageTemperature'] - df_jan_z['baseline_mean']) / df_jan_z['baseline_std']
df_jan_z.head()

> **Your Task (Part B — Count Extremes):**  
> 1. Define **extreme cold** as z < −2 and **extreme warm** as z > 2.  
> 2. Count how many extreme cold and extreme warm January events occurred in the **recent period (1980–2013)** vs. the **baseline period (1900–1950)**.  
> 3. Display the counts in a table or print them clearly.
>
> **Hint:** Filter by year range, then use `(df_jan['zscore'] < -2).sum()` and similar.

In [ ]:
baseline_z = df_jan_z[(df_jan_z['year'] >= 1900) & (df_jan_z['year'] <= 1950)]
recent_z = df_jan_z[(df_jan_z['year'] >= 1980) & (df_jan_z['year'] <= 2013)]

stats_list = []
for name, data in [('Baseline (1900-1950)', baseline_z), ('Recent (1980-2013)', recent_z)]:
    cold_count = (data['zscore'] < -2).sum()
    warm_count = (data['zscore'] > 2).sum()
    stats_list.append({'Period': name, 'Extreme Cold (z < -2)': cold_count, 'Extreme Warm (z > 2)': warm_count, 'Total Months': len(data)})

pd.DataFrame(stats_list)

> **Interpretation:**  
> Are extreme cold Januaries (z < −2) becoming **more or less common** in the recent period? What about extreme warm Januaries (z > 2)? What does this tell us about whether cold winters are disappearing?

**Answer:** 
- In the recent period (1980-2013), **extreme cold Januaries (z < -2) are becoming less common** (2 occurrences out of 272 months vs 6 in 408 baseline months, a decrease in frequency).
- **Extreme warm Januaries (z > 2) are becoming much more common** (26 occurrences in the recent period vs 13 in the baseline, despite the recent period being shorter).
- This strongly suggests that cold winters are "disappearing" or at least becoming rarer, while warmer winters are becoming much more frequent, mirroring the warming trend seen in summers.

---
## Task 4: Hypothesis Test [PRACTICE]

> **Your Task:**  
> 1. Run a **two-sample t-test** comparing Istanbul's January temperatures between **1900–1950** (old era) and **1980–2013** (recent era).  
> 2. State your **null hypothesis** ($H_0$) and **alternative hypothesis** ($H_a$) before running the test.  
> 3. Report the **t-statistic** and **p-value**.  
> 4. Interpret the result at the $\alpha = 0.05$ significance level.
>
> **Hint:** Use `stats.ttest_ind(old_era_temps, new_era_temps)`. Remember from lecture: if p < 0.05, we reject $H_0$.

In [ ]:
# H0: Mean Jan temp (1900-1950) = Mean Jan temp (1980-2013)
# Ha: Mean Jan temp (1900-1950) != Mean Jan temp (1980-2013)

t_stat, p_val = stats.ttest_ind(recent, early)
print(f"T-statistic: {t_stat:.4f}")
print(f"P-value: {p_val:.4f}")

> **Interpretation:**  
> What do you conclude? Is the change in Istanbul's January temperatures statistically significant? How does this compare to the **August result from lecture** — is the warming signal stronger in summer or winter?

**Answer:** 
- With a p-value of approx **0.0896**, we **fail to reject the null hypothesis** at the alpha = 0.05 level (since 0.0896 > 0.05).
- While there is an observed increase in mean temperature (+0.72°C), it is not statistically significant in this specific test.
- Compared to the **August result from lecture** (where p < 0.0001 and the shift was ~1.1°C), the warming signal appears **stronger and more statistically robust in summer (August)** than in winter (January) for Istanbul. This might be due to the higher naturally occurring variability (std dev) in winter temperatures, which makes it harder to distinguish a climate signal from noise.

---
## Bonus: Compare Across Cities [PRACTICE]

> **Your Task (if time permits):**  
> 1. Pick **2–3 additional cities** and run the same January t-test (1900–1950 vs. 1980–2013).  
> 2. Create a **summary table** with columns: City, Old Mean, New Mean, Difference, t-statistic, p-value, Significant?  
> 3. Do all cities show the same pattern?
>
> **Hint:** Use a loop over cities and collect results into a list of dictionaries, then convert to a DataFrame.

In [ ]:
results = []
for city in ['London', 'Tokyo', 'New York']:
    city_jan = df_jan[df_jan['CityLabel'] == city]
    c_early = city_jan[(city_jan['year'] >= 1900) & (city_jan['year'] <= 1950)]['AverageTemperature']
    c_recent = city_jan[(city_jan['year'] >= 1980) & (city_jan['year'] <= 2013)]['AverageTemperature']
    
    t, p = stats.ttest_ind(c_recent, c_early)
    results.append({
        'City': city,
        'Old Mean': c_early.mean(),
        'New Mean': c_recent.mean(),
        'Difference': c_recent.mean() - c_early.mean(),
        't-statistic': t,
        'p-value': p,
        'Significant?': p < 0.05
    })

pd.DataFrame(results)

**Note:** Tokyo shows a highly significant warming trend in January (p=0.0012), while London and New York show less significant results, highlighting how climate change impacts different regions with varying intensity.

---

### Questions?

**Dr. Eyuphan Koc**  
eyuphan.koc@bogazici.edu.tr